# Prompt Templates

Um prompt para um modelo de linguagem é um conjunto de instruções ou entradas fornecidas por um usuário para guiar a resposta do modelo, ajudando-o a entender o contexto e gerar uma saída baseada em linguagem relevante e coerente, como responder a perguntas, completar frases ou participar de uma conversa.

In [1]:
from langchain_openai.llms import OpenAI

llm = OpenAI()

In [2]:
# Importar Bibliotecas Necessárias
import os
from dotenv import load_dotenv
import requests
from langchain_community.llms import Ollama

# Carregar variáveis de ambiente
load_dotenv()

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [3]:
# Verificar conexão com Ollama
OLLAMA_BASE_URL = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    if response.status_code == 200:
        modelos = response.json()
        print("✅ Conexão com Ollama estabelecida!")
        print(f"\n📦 Modelos disponíveis:")
        for modelo in modelos.get('models', []):
            print(f"  - {modelo['name']}")
    else:
        print("❌ Erro ao conectar com Ollama")
except requests.exceptions.ConnectionError:
    print("❌ Não foi possível conectar ao Ollama. Certifique-se que está rodando em Docker")

✅ Conexão com Ollama estabelecida!

📦 Modelos disponíveis:
  - llama3.2:1b
  - llama3.1:8b
  - llama3.2:latest


In [4]:
# Inicializar LLM com Ollama
# Usando o modelo llama3.2:1b que foi baixado
llm = Ollama(
    base_url=OLLAMA_BASE_URL,
    model="llama3.2:1b"
)

print("✅ LLM Ollama inicializado!")
print(f"Modelo: llama3.2:1b")

✅ LLM Ollama inicializado!
Modelo: llama3.2:1b


C:\Users\Campeao Lub\AppData\Local\Temp\ipykernel_9812\1426091000.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [11]:
# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template('''
Responda a seguinte pergunta do usuário:
{pergunta}                                   
''')
prompt_template

PromptTemplate(input_variables=['pergunta'], input_types={}, partial_variables={}, template='\nResponda a seguinte pergunta do usuário:\n{pergunta}                                   \n')

In [12]:
print(prompt_template.format(pergunta='O que é um buraco negro?'))


Responda a seguinte pergunta do usuário:
O que é um buraco negro?                                   



In [5]:
# Versão antiga 
# from langchain.prompts import PromptTemplate

#versão mais atual 
from langchain_core.prompts import PromptTemplate


prompt_template = PromptTemplate.from_template('''
Responda a seguinte pergunta do usuário em até {n_palvras} palavras:
{pergunta}                                   
''')
prompt_template.format(n_palvras=10, pergunta='O que é um buraco negro?')

'\nResponda a seguinte pergunta do usuário em até 10 palavras:\nO que é um buraco negro?                                   \n'

In [6]:
prompt_template = PromptTemplate.from_template('''
Responda a seguinte pergunta do usuário em até {n_palvras} palavras:
{pergunta}                                   
''', partial_variables={'n_palvras': 10})
prompt_template.format(pergunta='O que é um buraco negro?')

'\nResponda a seguinte pergunta do usuário em até 10 palavras:\nO que é um buraco negro?                                   \n'

In [7]:
prompt_template.format(n_palvras=5, pergunta='O que é um buraco negro?')

'\nResponda a seguinte pergunta do usuário em até 5 palavras:\nO que é um buraco negro?                                   \n'

## Composing prompts | Unindo múltiplos prompts

In [13]:
# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate

template_word_count = PromptTemplate.from_template('''
Responda a pergunta em até {n_palavras} palavras.''')

template_lingua = PromptTemplate.from_template('''
Retorn a resposta na {lingua}.''')

template_final = (
    template_word_count
    + template_lingua
    + '\nResponda a pergunta seguinte seguindo as instruções: {pergunta}'
)

template_final.template

'\nResponda a pergunta em até {n_palavras} palavras.\nRetorn a resposta na {lingua}.\nResponda a pergunta seguinte seguindo as instruções: {pergunta}'

In [14]:
prompt = template_final.format(n_palavras=10, lingua='espanhol', pergunta='O que é uma estrela?')
llm.invoke(prompt)

'Uma estrela é un globo de gaseo'

In [15]:
prompt = template_final.format(n_palavras=10, lingua='inglês', pergunta='O que é uma estrela?')
print(prompt)


Responda a pergunta em até 10 palavras.
Retorn a resposta na inglês.
Responda a pergunta seguinte seguindo as instruções: O que é uma estrela?


## Templates para Chat

In [16]:
# from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate


chat_template = ChatPromptTemplate.from_template('Essa é a minha dúvida: {duvida}')
chat_template.format_messages(duvida='Quem sou eu?')

[HumanMessage(content='Essa é a minha dúvida: Quem sou eu?', additional_kwargs={}, response_metadata={})]

In [17]:
# from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ('system', 'Você é um assistente engraçado e se chama {nome_assistente}'),
        ('human', 'Olá, como vai?'),
        ('ai', 'Melhor agora! como posso ajudá-lo?'),
        ('human', '{pergunta}')
    ]
)

chat_template

ChatPromptTemplate(input_variables=['nome_assistente', 'pergunta'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['nome_assistente'], input_types={}, partial_variables={}, template='Você é um assistente engraçado e se chama {nome_assistente}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Olá, como vai?'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Melhor agora! como posso ajudá-lo?'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pergunta'], input_types={}, partial_variables={}, template='{pergunta}'), additional_kwargs={})])

In [18]:
chat_template.format_messages(nome_assistente='Asimo', pergunta='Qual o seu nome?')

[SystemMessage(content='Você é um assistente engraçado e se chama Asimo', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Olá, como vai?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Melhor agora! como posso ajudá-lo?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Qual o seu nome?', additional_kwargs={}, response_metadata={})]

In [21]:
# from langchain_openai.chat_models import ChatOpenAI
from langchain_ollama import ChatOllama

chat = ChatOllama(base_url=OLLAMA_BASE_URL, model="llama3.2:1b")

responsta = chat.invoke(chat_template.format_messages(nome_assistente='Asimo', pergunta='Qual o seu nome?'))


display(responsta)

AIMessage(content='Meu nome é Asimo, mas você pode me chamar de... bem, qualquer coisa que você preferir! Eu sou aqui para ajudar com qualquer coisa, seja uma pergunta, uma dúvida ou até mesmo uma brincadeira. Estou aqui para te falar!', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-03T20:16:40.0947281Z', 'done': True, 'done_reason': 'stop', 'total_duration': 81541354900, 'load_duration': 77629131100, 'prompt_eval_count': 72, 'prompt_eval_duration': 642375400, 'eval_count': 61, 'eval_duration': 3245002000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019cb557-3a18-7051-9c8b-c986b16aad21-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 61, 'total_tokens': 133})

## Templates de Few-shot prompting para llm

In [22]:
exemplos = [
    {"pergunta": "Quem viveu mais tempo, Muhammad Ali ou Alan Turing?", 
     "resposta": 
     """São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quantos anos Muhammad Ali tinha quando morreu? 
Resposta intermediária: Muhammad Ali tinha 74 anos quando morreu. 
Pergunta de acompanhamento: Quantos anos Alan Turing tinha quando morreu? 
Resposta intermediária: Alan Turing tinha 41 anos quando morreu. 
Então a resposta final é: Muhammad Ali 
""", 
    }, 
    {"pergunta": "Quando nasceu o fundador do craigslist?", 
     "resposta": 
"""São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quem foi o fundador do craigslist? 
Resposta intermediária: O craigslist foi fundado por Craig Newmark. 
Pergunta de acompanhamento: Quando nasceu Craig Newmark? 
Resposta intermediária: Craig Newmark nasceu em 6 de dezembro de 1952. 
Então a resposta final é: 6 de dezembro de 1952 
""", 
    }, 
    {"pergunta": "Quem foi o avô materno de George Washington?",
     "resposta": 
"""São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quem foi a mãe de George Washington? 
Resposta intermediária: A mãe de George Washington foi Mary Ball Washington. 
Pergunta de acompanhamento: Quem foi o pai de Mary Ball Washington? 
Resposta intermediária: O pai de Mary Ball Washington foi Joseph Ball. 
Então a resposta final é: Joseph Ball 
""", 
    },
    {"pergunta": "Os diretores de Jaws e Casino Royale são do mesmo país?", 
     "resposta": 
"""São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quem é o diretor de Jaws? 
Resposta Intermediária: O diretor de Jaws é Steven Spielberg. 
Pergunta de acompanhamento: De onde é Steven Spielberg? 
Resposta Intermediária: Estados Unidos. 
Pergunta de acompanhamento: Quem é o diretor de Casino Royale? 
Resposta Intermediária: O diretor de Casino Royale é Martin Campbell. 
Pergunta de acompanhamento: De onde é Martin Campbell? 
Resposta Intermediária: Nova Zelândia. 
Então a resposta final é: Não 
""",
    },
]

In [24]:
# from langchain.prompts.few_shot import FewShotPromptTemplate
# from langchain.prompts.prompt import PromptTemplate

from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate


example_prompt = PromptTemplate(
    input_variables=['pergunta', 'resposta'],
    template='Pergunta {pergunta}\n{resposta}'
)

example_prompt.format(**exemplos[0]).split('\n')


['Pergunta Quem viveu mais tempo, Muhammad Ali ou Alan Turing?',
 'São necessárias perguntas de acompanhamento aqui: Sim. ',
 'Pergunta de acompanhamento: Quantos anos Muhammad Ali tinha quando morreu? ',
 'Resposta intermediária: Muhammad Ali tinha 74 anos quando morreu. ',
 'Pergunta de acompanhamento: Quantos anos Alan Turing tinha quando morreu? ',
 'Resposta intermediária: Alan Turing tinha 41 anos quando morreu. ',
 'Então a resposta final é: Muhammad Ali ',
 '']

In [29]:
prompt = FewShotPromptTemplate(
    examples=exemplos,
    example_prompt=example_prompt,
    suffix='Pergunta: {input}',
    input_variables=['input']
)
# prompt.format(input='Quem viveu mais tempo, Muhammad Ali ou Alan Turing?').split('\n')

In [31]:
print(prompt.format(input='Quem fez mais gols, Romário ou Pelé?'))

Pergunta Quem viveu mais tempo, Muhammad Ali ou Alan Turing?
São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quantos anos Muhammad Ali tinha quando morreu? 
Resposta intermediária: Muhammad Ali tinha 74 anos quando morreu. 
Pergunta de acompanhamento: Quantos anos Alan Turing tinha quando morreu? 
Resposta intermediária: Alan Turing tinha 41 anos quando morreu. 
Então a resposta final é: Muhammad Ali 


Pergunta Quando nasceu o fundador do craigslist?
São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quem foi o fundador do craigslist? 
Resposta intermediária: O craigslist foi fundado por Craig Newmark. 
Pergunta de acompanhamento: Quando nasceu Craig Newmark? 
Resposta intermediária: Craig Newmark nasceu em 6 de dezembro de 1952. 
Então a resposta final é: 6 de dezembro de 1952 


Pergunta Quem foi o avô materno de George Washington?
São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: 

In [33]:
llm.invoke(prompt.format(input='Quem fez mais gols, Romário ou Pelé?')).split('\n') 

['Entendo que você deseja uma resposta final para suas perguntas, mas é importante notar que as informações fornecidas em cada uma delas não são precisas ou precisas em um contexto histórico específico.',
 '',
 '- Muhammad Ali: Sua idade quando morreu foi de 74 anos, mas o fato de ele ter vivido mais tempo do que Alan Turing é uma afirmação que pode não ser precisa. Além disso, não há registros confiáveis que indiquem a idade de Muhammad Ali quando morreu.',
 '- Craig Newmark: Sua data de nascimento foi de 6 de dezembro de 1952, o que o torna um membro antigo da internet, contribuindo para a criação do craigslist.',
 '- George Washington: A mãe de George Washington foi Mary Ball Washington.',
 '- Joseph Ball: A mãe de Joseph Ball foi Anne Ball.',
 '- Diretores de Jaws e Casino Royale: Ambos são filhos de diferentes pessoas, mas não há registros comprovados de que ambos tenham trabalhado como diretores de filmes.',
 '- Estados Unidos e Nova Zelândia: Não há registros comprovados de que 

## Templates de Few-shot prompting para chat

In [34]:
from langchain_ollama import ChatOllama

chat = ChatOllama(base_url=OLLAMA_BASE_URL, model="llama3.2:1b")



In [35]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate

example_prompt = ChatPromptTemplate.from_messages(
    [('human', '{pergunta}'),
     ('ai', '{resposta}')]
)

print(example_prompt.format_messages(**exemplos[0]))

[HumanMessage(content='Quem viveu mais tempo, Muhammad Ali ou Alan Turing?', additional_kwargs={}, response_metadata={}), AIMessage(content='São necessárias perguntas de acompanhamento aqui: Sim. \nPergunta de acompanhamento: Quantos anos Muhammad Ali tinha quando morreu? \nResposta intermediária: Muhammad Ali tinha 74 anos quando morreu. \nPergunta de acompanhamento: Quantos anos Alan Turing tinha quando morreu? \nResposta intermediária: Alan Turing tinha 41 anos quando morreu. \nEntão a resposta final é: Muhammad Ali \n', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [ ]:
few_shot_template = FewShotChatMessagePromptTemplate(
    examples=exemplos,
    example_prompt=example_prompt
)

prompt_template = ChatPromptTemplate.from_messages(
    [
    few_shot_template,
    ('human', '{input}')
    ]
)

prompt = prompt_template.format_messages(input='Quem fez mais gols, Messi ou Pelé?')
prompt

[HumanMessage(content='Quem viveu mais tempo, Muhammad Ali ou Alan Turing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='São necessárias perguntas de acompanhamento aqui: Sim. \nPergunta de acompanhamento: Quantos anos Muhammad Ali tinha quando morreu? \nResposta intermediária: Muhammad Ali tinha 74 anos quando morreu. \nPergunta de acompanhamento: Quantos anos Alan Turing tinha quando morreu? \nResposta intermediária: Alan Turing tinha 41 anos quando morreu. \nEntão a resposta final é: Muhammad Ali \n', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Quando nasceu o fundador do craigslist?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='São necessárias perguntas de acompanhamento aqui: Sim. \nPergunta de acompanhamento: Quem foi o fundador do craigslist? \nResposta intermediária: O craigslist foi fundado por Craig Newmark. \nPergunta de acompanhamento: Quando nasceu Craig Newmark? \nRespo

In [44]:
prompt = prompt_template.format_messages(input='Quem fez mais gols, Messi ou Pelé?')

ult_resposta = chat.invoke(prompt)

In [43]:
print(ult_resposta.content )

São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quem foi o maior goleiro da história do futebol? 
Resposta intermediária: Pelé. 
Pergunta de acompanhamento: Quem foi o maior jogador de futebol? 
Resposta intermediária: Romário. 
Então a resposta final é: Romário


In [45]:
print(ult_resposta.content )

São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quem foi o jogador de futebol mais icônico do mundo? 
Resposta intermediária: Pelé. 
Pergunta de acompanhamento: Quem foi o jogador de futebol mais icônico do mundo? 
Resposta intermediária: Lionel Messi. 
Então a resposta final é: Lionel Messi
